# RAG Evaluation with RAGAS

**Intro to RAG evaluation · The RAGAS metrics · Preparing a dataset · Evaluating a RAG app**

---

### The question this notebook answers

You built a RAG chatbot. Your boss asks:

> *"Is it good?"*

Right now your honest answer is *"I tried about ten questions and they looked fine."*

That is not an answer. It does not scale, it cannot be repeated, and it cannot tell you
whether the change you shipped yesterday made things better or worse.

**RAGAS turns "it looked fine" into a number.**

---

# Part 1 - Introduction to RAG Evaluation and RAGAS

### Why evaluating RAG is harder than normal testing

A normal function has one right answer, so testing is easy:

```
   add(2, 2)  ->  4        correct or not, no argument
```

A RAG app has **many** acceptable answers:

```
   "How long do refunds take?"
      ->  "Refunds take 7 business days."
      ->  "You will get your money back within a week of business days."
      ->  "Seven working days."
```

All three are right. `assert answer == expected` is useless here.

---

### The key insight: a RAG app has TWO halves

This is the most important idea in the notebook.

```
                    HALF 1: RETRIEVER                  HALF 2: GENERATOR
                 (find the right documents)        (write an answer from them)

   question ───────────► search ───────────► documents ───────────► LLM ───────────► answer
                            |                                        |
                            |                                        |
                    can fail by bringing              can fail by ignoring good
                    the wrong documents               documents and making things up
```

**They fail in completely different ways, and the fix is completely different.**

| What you see                                        | Which half broke | What you fix                    |
| --------------------------------------------------- | ---------------- | ------------------------------- |
| Answer is wrong, and the documents were wrong too   | Retriever        | chunking, embeddings, search, k |
| Answer is wrong, but the right documents were there | Generator        | the prompt, the model           |

A single "accuracy" score cannot tell these apart. That is exactly the gap RAGAS fills:
**it scores each half separately.**

---

### So what is RAGAS?

> **RAGAS is a library that scores your RAG answers from 0 to 1, using an LLM as the judge.**

You give it what your app did (question, documents it found, answer it wrote).
RAGAS asks a second LLM careful questions about that, and returns numbers.

```
   your RAG output  ──►  RAGAS  ──►  faithfulness      0.95
                          |          answer relevancy  0.88
                     (judge LLM)     context precision 0.50   <- the weak spot
                                     context recall    0.45
```

### Two things to be clear about up front

1. **The judge is an LLM, so evaluation costs money and takes time.**
   Every metric on every row is at least one LLM call. 5 questions x 4 metrics is
   20-plus calls. Start small.
2. **Scores are not exact truth.** They are a consistent, repeatable opinion.
   Their real value is **comparing** version A against version B, not the absolute number.

---

# Part 2 - Understanding RAGAS Metrics

### The four you should learn first

| Metric                | The question it asks                                  | Which half it tests |
| --------------------- | ----------------------------------------------------- | ------------------- |
| **Faithfulness**      | Is the answer actually supported by the documents?    | Generator           |
| **Answer relevancy**  | Does the answer address the question that was asked?  | Generator           |
| **Context precision** | Were the retrieved documents useful, or mostly noise? | Retriever           |
| **Context recall**    | Did we retrieve everything needed to answer?          | Retriever           |

All four return **0 to 1, higher is better.**

---

### Reading a low score

This table is the reason you run RAGAS at all.

| Low score         | What it means in plain English                              | Where to look                                  |
| ----------------- | ----------------------------------------------------------- | ---------------------------------------------- |
| Faithfulness      | The model is **making things up**, even with good documents | the prompt, the model                          |
| Answer relevancy  | The answer wanders or dodges the question                   | the prompt                                     |
| Context precision | You retrieved a lot of junk alongside the useful bit        | chunk size, k, reranking                       |
| Context recall    | The needed information was **never retrieved**              | chunking, embeddings, the documents themselves |

> **Worked example.** Faithfulness 0.4 with context recall 0.95 means the documents were
> there and the model ignored them. Do not touch your retriever. Fix the prompt.

---

### The split that decides your whole workflow: does it need a reference answer?

A **reference** is a human-written correct answer, also called ground truth.
Writing them is slow, so this matters:

| Metric                             | Needs a reference? |
| ---------------------------------- | ------------------ |
| Faithfulness                       | **No**             |
| Answer relevancy                   | **No**             |
| Context precision (with reference) | Yes                |
| Context recall                     | Yes                |
| Factual correctness                | Yes                |
| Semantic similarity                | Yes                |

**Why you care:** the two "No" metrics can run on **live production traffic**, where nobody
has written a correct answer. The "Yes" metrics need a prepared test set.

So the usual setup is:

```
   test set with references   ->  all metrics       ->  run before each release
   live production traffic    ->  the 2 no-reference metrics  ->  run continuously
```

---

# Part 3 - Preparing a Dataset for RAGAS Evaluation

### Step 1 - Setup

### Step 2 - The four field names

A RAGAS row has four fields. **The names are strict.**

| Field                | What goes in it                                        | Comes from      |
| -------------------- | ------------------------------------------------------ | --------------- |
| `user_input`         | the question                                           | you             |
| `retrieved_contexts` | the documents your app found, as a **list of strings** | your retriever  |
| `response`           | the answer your app produced                           | your LLM        |
| `reference`          | the correct answer, written by a human                 | you, in advance |

> **This is the number one place RAGAS code breaks.** Older tutorials use the old names.
> If you copy code from a blog post and get a confusing error, check this table first.
>
> | Old name (do not use) | Current name         |
> | --------------------- | -------------------- |
> | `question`            | `user_input`         |
> | `contexts`            | `retrieved_contexts` |
> | `answer`              | `response`           |
> | `ground_truth`        | `reference`          |


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model_name="openai/gpt-oss-20b",
    api_key=os.environ.get("LLM_API_KEY"),
    base_url=os.environ.get("LLM_BASE_URL"),
    temperature=0.1, 
)

In [3]:
from langchain_ollama import OllamaEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

documents = [
    """
    REFUND ELIGIBILITY

    Customers may request a refund within 30 days of the purchase date.
    To be eligible for a refund, the product must be unused and in its
    original condition. Digital products may be eligible for a refund
    only if they have not been substantially used or downloaded.
    """,

    """
    NON-REFUNDABLE ITEMS

    The following items are non-refundable:
    - Gift cards
    - Discounted or clearance items
    - Personalized products
    - Products damaged by the customer
    - Services that have already been fully completed
    """,

    """
    HOW TO REQUEST A REFUND

    To request a refund, contact our customer support team with your
    order number, registered email address, and reason for the refund.
    Refund requests are typically reviewed within 3 to 5 business days.
    """,

    """
    REFUND PROCESSING TIME

    Once a refund is approved, the refund will be processed to the
    original payment method. Credit and debit card refunds may take
    5 to 10 business days to appear. Bank transfer refunds may take
    up to 7 business days.
    """,

    """
    SUBSCRIPTION REFUND POLICY

    Customers may cancel their subscription at any time. Monthly
    subscription payments are generally non-refundable after the
    billing period has started. Annual subscriptions may be eligible
    for a partial refund if cancelled within 14 days of renewal.
    """,

    """
    DAMAGED OR INCORRECT PRODUCTS

    If you receive a damaged, defective, or incorrect product, contact
    customer support within 7 days of delivery. You may be eligible for
    a replacement or a full refund. Supporting photographs may be
    required to process the request.
    """,

    """
    LATE REFUND REQUESTS

    Refund requests submitted after the standard 30-day refund period
    are normally not accepted. Exceptions may be considered for
    technical errors, duplicate charges, or other special circumstances.
    """
]

embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")
vector_store = InMemoryVectorStore.from_texts(documents, embedding=embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

found = retriever.invoke("I need my money back, how long does it take?")

for doc in found:
    print("-", doc.page_content)

- 
    REFUND PROCESSING TIME

    Once a refund is approved, the refund will be processed to the
    original payment method. Credit and debit card refunds may take
    5 to 10 business days to appear. Bank transfer refunds may take
    up to 7 business days.
    
- 
    HOW TO REQUEST A REFUND

    To request a refund, contact our customer support team with your
    order number, registered email address, and reason for the refund.
    Refund requests are typically reviewed within 3 to 5 business days.
    


In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([    
    ("system", "You are a support assisstant Answer only using the context below. \n\n{context}"),
    ("human", "{question}"),
])

In [5]:
test_questions = [
    "How long do refund take ?",
    "How fast is express shipping ?",
    "When is the support team is available ?",
    "How do i cancel my subscription ?"
]

referrence_answers = [
    "Refund are processed within 7 days",
    "Express shipping takes 1 business day.",
    "Support team is available Monday to Friday, 9am to 6pm IST",
    "You can cancel anytime from the Settings, and billing stop at the end of the cycle."
]

In [6]:
from email import message
from opentelemetry import context


rows = []

for question, referrence in zip(test_questions, referrence_answers):
    docs = retriever.invoke(question)

    contexts = []
    for doc in docs:
        contexts.append(doc.page_content)

    context_text = "".join(contexts)
    messages = prompt.format_messages(context=context_text, question=question)

    answer = llm.invoke(messages).content

    rows.append({
        "user_input": question,
        "retrieved_contexts": contexts,
        "response": answer,
        "reference": referrence
    })
print("Done")

Done


In [7]:
from ragas import EvaluationDataset

dataset = EvaluationDataset.from_list(rows)
print("Length of the dataset: ",len(dataset))
print(dataset[0])

/home/balaji/LLM/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Length of the dataset:  4
user_input='How long do refund take ?' retrieved_contexts=['\n    REFUND PROCESSING TIME\n\n    Once a refund is approved, the refund will be processed to the\n    original payment method. Credit and debit card refunds may take\n    5 to 10 business days to appear. Bank transfer refunds may take\n    up to 7 business days.\n    ', '\n    HOW TO REQUEST A REFUND\n\n    To request a refund, contact our customer support team with your\n    order number, registered email address, and reason for the refund.\n    Refund requests are typically reviewed within 3 to 5 business days.\n    '] reference_contexts=None retrieved_context_ids=None reference_context_ids=None response='Refunds typically take:\n\n- **Credit or debit card**: 5 to 10 business days to appear on your statement.  \n- **Bank transfer**: up to 7 business days to be credited to your account.  \n\n(Refund requests are usually reviewed within 3 to 5 business days before the refund is processed.)' multi_re

In [8]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

import os 

judge_chat_model = ChatOpenAI(
    model=os.getenv("LLM_MODEL"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
    temperature=0,
)

judge_llm = LangchainLLMWrapper(judge_chat_model)
judge_embeddings = LangchainEmbeddingsWrapper(OllamaEmbeddings(model="nomic-embed-text:latest"))

/tmp/ipykernel_2864290/3052195623.py:13: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(judge_chat_model)
/tmp/ipykernel_2864290/3052195623.py:14: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_embeddings = LangchainEmbeddingsWrapper(OllamaEmbeddings(model="nomic-embed-text:latest"))


In [9]:
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy, 
    LLMContextPrecisionWithReference, 
    LLMContextRecall
)
from ragas import evaluate

metrics = [
    # Faithfulness(),
    ResponseRelevancy(), 
    # LLMContextPrecisionWithReference(), 
    # LLMContextRecall()
]
result = evaluate(
    dataset = dataset,
    metrics = metrics,
    llm = judge_llm,
    embeddings = judge_embeddings
)
print(result)

/tmp/ipykernel_2864290/3618434155.py:1: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
/tmp/ipykernel_2864290/3618434155.py:1: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import (
/tmp/ipykernel_2864290/3618434155.py:1: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import (
/tmp/ipykernel_2864290/3618434155.py:1: DeprecationWarning: Importing LLMContextRecall from 'ra

{'answer_relevancy': 0.4315}


In [10]:
scores = result.to_pandas()
scores

,user_input,retrieved_contexts,response,reference,answer_relevancy
0,How long do refund take ?,[\n REFUND PROCESSING TIME\n\n Once a re...,Refunds typically take:\n\n- **Credit or debit...,Refund are processed within 7 days,0.799306
1,How fast is express shipping ?,[\n REFUND PROCESSING TIME\n\n Once a re...,"I’m sorry, but I don’t have that information.",Express shipping takes 1 business day.,0.000000
2,When is the support team is available ?,[\n DAMAGED OR INCORRECT PRODUCTS\n\n If...,"I’m sorry, but the information you’ve provided...","Support team is available Monday to Friday, 9a...",0.000000
3,How do i cancel my subscription ?,[\n SUBSCRIPTION REFUND POLICY\n\n Custo...,You can cancel your subscription at any time. ...,"You can cancel anytime from the Settings, and ...",0.926862
